In [1]:
from pinky_lcd.pinky_lcd import LCD
from pinkylib import LED, Camera, Motor
from PIL import Image, ImageSequence
from IPython.display import display
import time, io, base64, json

In [2]:
led = LED()
lcd = LCD()
motor = Motor(); motor.enable_motor()

모터 #1를 성공적으로 '속도 제어 모드'로 설정했습니다.
모터 #2를 성공적으로 '속도 제어 모드'로 설정했습니다.


In [3]:
EMOTIONS = ['angry','basic','bored','fun','happy','hello','interest','sad']

In [4]:
RED=(255,0,0); ORANGE=(255,127,0); YELLOW=(255,255,0)
GREEN=(0,255,0); BLUE=(0,0,255); INDIGO=(75,0,130)
VIOLET=(148,0,211); WHITE=(255,255,255)

RAINBOW = [RED,ORANGE,YELLOW,GREEN,BLUE,INDIGO,VIOLET]

In [5]:
def play_rainbow_once(dt=0.1):
    for c in RAINBOW:
        led.fill(c); time.sleep(dt)

In [6]:
play_rainbow_once(0.15)

In [7]:
play_rainbow_once(0.5)

In [8]:
led.__dir__()

['_strip',
 'count',
 '__module__',
 '__init__',
 '__enter__',
 '__exit__',
 'show',
 'clear',
 'set_pixel',
 'set_brightness',
 'get_brightness',
 'get_pixel_color',
 'fill',
 'numPixels',
 '_wheel',
 'color_wipe',
 'theater_chase',
 'rainbow',
 'rainbowCycle',
 'theaterChaseRainbow',
 'close',
 '__dict__',
 '__weakref__',
 '__doc__',
 '__new__',
 '__repr__',
 '__hash__',
 '__str__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__getstate__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [9]:
led.clear()

In [10]:
def play_led_emotion(e):
    if e=="angry":   led.fill(RED)
    elif e=="happy": led.fill(YELLOW)
    elif e=="fun":   play_rainbow_once(0.08)
    elif e=="sad":   led.fill(BLUE)
    elif e=="bored": led.fill(INDIGO)
    elif e=="interest": led.fill(GREEN)
    elif e=="hello":
        led.fill(WHITE); time.sleep(0.2)
        led.fill((0,0,0)); time.sleep(0.1); led.fill(WHITE)
    else: led.fill(WHITE)

In [11]:
play_led_emotion("hello")

In [12]:
led.clear()

In [13]:
import os 

emotion_folder_path = '/home/pinky/pinky_pro/src/pinky_pro/pinky_emotion/emotion/'
files = os.listdir(emotion_folder_path)

print(files)

['angry.gif', 'basic.gif', 'bored.gif', 'hello.gif', 'interest.gif', 'fun.gif', 'happy.gif', 'sad.gif']


In [14]:
print("Start GIF")

gif = Image.open(emotion_folder_path + "fun.gif")
for frame in ImageSequence.Iterator(gif):
    lcd.img_show(frame)

lcd.clear()

print("End GIF")

Start GIF
End GIF


In [15]:
def play_emotion(emotion):
    emotion_folder_path = '/home/pinky/pinky_pro/src/pinky_pro/pinky_emotion/emotion/'
    emotion_path = f"{emotion_folder_path}/{emotion}.gif"
    gif = Image.open(emotion_path)
    
    n = 0
    for frame in ImageSequence.Iterator(gif):
        lcd.img_show(frame)
        n += 1
        if n > 20:
            break
        
    print("End Emotion")

In [16]:
def motor_wiggle_lr(c=3, sp=20, dt=0.3):
    for _ in range(c):
        motor.move(sp,-sp); time.sleep(dt)
        motor.move(-sp,sp); time.sleep(dt)
    motor.stop()

In [17]:
def motor_wiggle_fb(c=2, sp=5, dt=0.1):
    for _ in range(c):
        motor.move(sp,sp); time.sleep(dt)
        motor.move(-sp,-sp); time.sleep(dt)
    motor.stop()

In [18]:
def motor_nod_forward(sp=10, dt=0.1):
    motor.move(sp,sp); time.sleep(dt); motor.stop()

In [19]:
def play_motor_emotion(e):
    if e in ["happy","fun","hello"]:
        motor_wiggle_lr()
    elif e=="sad":
        motor_wiggle_fb()
    elif e=="angry":
        motor_wiggle_fb(c=1, sp=70, dt=0.2)
    elif e=="interest":
        motor_nod_forward(sp=35, dt=0.25)
    elif e=="bored":
        motor_wiggle_lr(c=1, sp=20, dt=0.5)
    else:
        motor.stop()

In [20]:
play_motor_emotion("angry")

In [22]:
play_motor_emotion("interest")

In [21]:
def capture_one_frame_bytes(cam):
    frame = cam.get_frame()        # 환경에 맞게 메서드 이름 조정
    img = Image.fromarray(frame[:, :, ::-1])  # BGR -> RGB
    buf = io.BytesIO(); img.save(buf, format="JPEG")
    return buf.getvalue()

In [22]:
def test_camera_once():
    cam = Camera(); cam.start()
    img_bytes = capture_one_frame_bytes(cam)
    display(Image.open(io.BytesIO(img_bytes)))
    cam.close()

In [ ]:
test_camera_once()

In [24]:
import os 
from dotenv import load_dotenv 
from openai import OpenAI  

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [25]:
SYSTEM_PROMPT = f"""
You are a helpful robot friend.

Always reply ONLY in JSON:
{{
    "assistant_message": "<reply in Korean>",
    "emotion": "<one of: {', '.join(EMOTIONS)}>"
}}

Infer the user's current emotion.
If unclear, use "basic".
No extra fields, no text outside JSON.
"""

In [26]:
def build_chat_messages(text, hist):
    msgs = [{"role":"system","content":SYSTEM_PROMPT}]
    msgs += hist
    msgs.append({"role":"user","content":text})
    return msgs

In [27]:
# def request_chat_emotion(msgs, model="gpt-4.1-mini"):
#     res = client.chat.completions.create(
#         model=model, messages=msgs, temperature=0.4
#     )
#     raw = res.choices[0].message.content
#     print("RAW:", raw)
#     try:
#         data = json.loads(raw)
#         msg = data.get("assistant_message","")
#         emo = data.get("emotion","basic")
#     except json.JSONDecodeError:
#         msg, emo = raw, "basic"
#     if emo not in EMOTIONS: emo = "basic"
#     return msg, emo, raw

def request_chat_emotion(msgs, model="gpt-5.5"):
    res = client.chat.completions.create(
        model=model, messages=msgs
    )
    raw = res.choices[0].message.content
    print("RAW:", raw)
    try:
        data = json.loads(raw)
        msg = data.get("assistant_message","")
        emo = data.get("emotion","basic")
    except json.JSONDecodeError:
        msg, emo = raw, "basic"
    if emo not in EMOTIONS: emo = "basic"
    return msg, emo, raw


In [28]:
def chat_with_emotion(text, history=None):
    if history is None: history = []
    msgs = build_chat_messages(text, history)
    reply, emo, raw = request_chat_emotion(msgs)
    history += [{"role":"user","content":text},
                {"role":"assistant","content":raw}]
    return reply, emo, history

In [56]:
history = []
print("대화 시작 (종료: quit/exit/종료)")

while True:
    user = input("You: ")
    if user.lower() in ["quit","exit","종료"]:
        break
    reply, emo, history = chat_with_emotion(user, history)
    print(f"REQ : {user}")
    print(f"Bot({emo}):", reply)
    play_led_emotion(emo)
    play_emotion(emo)
    play_motor_emotion(emo)


대화 시작 (종료: quit/exit/종료)


You:  종료


In [29]:
VISION_SYSTEM_PROMPT = f"""
You see a single face image.

Return ONLY JSON:
{{ "emotion": "<one of: {', '.join(EMOTIONS)}>" }}

Choose the best matching emotion from the face.
If unclear, use "basic".
"""


In [30]:
def encode_image_bytes(img_bytes: bytes) -> str:
    return base64.b64encode(img_bytes).decode("utf-8")


In [31]:
# def request_vision_raw_response(b64_img, model="gpt-4.1-mini"):
#     msg = [
#         {"role":"system","content":VISION_SYSTEM_PROMPT},
#         {"role":"user","content":[
#             {"type":"text","text":"Choose ONE emotion."},
#             {"type":"image_url",
#                 "image_url":{"url":f"data:image/jpeg;base64,{b64_img}"}}
#         ]}
#     ]
#     res = client.chat.completions.create(
#         model=model, messages=msg, temperature=0, max_completion_tokens=64
#     )
#     raw = res.choices[0].message.content
#     print("RAW:", raw)
#     return raw


def request_vision_raw_response(b64_img, model="gpt-5.5"):
    msg = [
        {"role":"system","content":VISION_SYSTEM_PROMPT},
        {"role":"user","content":[
            {"type":"text","text":"Choose ONE emotion."},
            {"type":"image_url",
                "image_url":{"url":f"data:image/jpeg;base64,{b64_img}"}}
        ]}
    ]
    res = client.chat.completions.create(
        model=model, messages=msg, max_completion_tokens=64
    )
    raw = res.choices[0].message.content
    print("RAW:", raw)
    return raw


In [32]:
def parse_vision_emotion(raw: str) -> str:
    try:
        data = json.loads(raw)
        emo = data.get("emotion","basic").strip()
    except json.JSONDecodeError:
        emo = "basic"
    if emo not in EMOTIONS: emo = "basic"
    return emo


In [33]:
def detect_emotion_from_image_bytes(img_bytes: bytes) -> str:
    b64 = encode_image_bytes(img_bytes)
    raw = request_vision_raw_response(b64)
    return parse_vision_emotion(raw)


In [34]:
def capture_image(cam):
    img_bytes = capture_one_frame_bytes(cam)
    display(Image.open(io.BytesIO(img_bytes)))
    return img_bytes

def process_emotion(img_bytes):
    emo = detect_emotion_from_image_bytes(img_bytes)
    print("Detected emotion:", emo)
    return emo


In [35]:
def run_camera_emotion_once():
    cam = Camera(); cam.start()
    img_bytes = capture_image(cam)
    emo = process_emotion(img_bytes)
    play_led_emotion(emo)
    play_emotion(emo)
    play_motor_emotion(emo)
    cam.close()


In [ ]:
run_camera_emotion_once()

In [42]:
led.clear()